# `metatomic.System` vs `torch.compile` / `jax.jit`

External notes for [metatensor/metatomic#325](https://github.com/metatensor/metatomic/pull/325). Not part of the PR.

`metatomic.System` is a ctypes wrapper around `mta_system_t`. This notebook checks:

1. Host-side construction and getters with numpy / torch / jax (including DLPack round-trips).
2. Whether **extracted arrays** can be fed into `torch.compile` / `jax.jit`.
3. Whether `System` itself can live **inside** a compiled/traced region.

Expected split: construct `System` in Python, then pass arrays into compiled kernels. Do not trace the wrapper.

In [1]:
import traceback

import jax
import jax.numpy as jnp
import numpy as np
import torch
import torch._dynamo as dynamo
from metatomic import System

jax.config.update("jax_enable_x64", True)

print("torch", torch.__version__)
print("jax", jax.__version__)
print("metatomic", System.__module__)


def report(label, fn):
    try:
        result = fn()
    except Exception as exc:
        print(f"FAIL  {label}")
        print(f"      {type(exc).__name__}: {exc}")
        return None
    print(f"OK    {label}: {result}")
    return result

torch 2.13.0
jax 0.11.1
metatomic metatomic


## 1. Host-side round-trips

Construct `System` from each backend and read `types` / `positions` / `cell` / `pbc` back.

In [2]:
def numpy_arrays():
    types = np.array([1, 4, 7, 10], dtype=np.int32)
    positions = np.array(
        [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0], [10.0, 11.0, 12.0]],
        dtype=np.float64,
    )
    cell = np.array(
        [[10.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 10.0]], dtype=np.float64
    )
    pbc = np.array([True, False, True])
    return types, positions, cell, pbc


np_types, np_positions, np_cell, np_pbc = numpy_arrays()
system_np = System("nm", np_types, np_positions, np_cell, np_pbc)
print("backend", system_np.arrays_backend)
print("positions\n", system_np.positions)
assert system_np.arrays_backend == "numpy"
assert isinstance(system_np.positions, np.ndarray)
assert float(system_np.positions[3, 0]) == 10.0

backend numpy
positions
 [[ 1.  2.  3.]
 [ 4.  5.  6.]
 [ 7.  8.  9.]
 [10. 11. 12.]]


In [3]:
th_types = torch.tensor([1, 4, 7, 10], dtype=torch.int32)
th_positions = torch.tensor(
    [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0], [10.0, 11.0, 12.0]],
    dtype=torch.float64,
)
th_cell = torch.tensor(
    [[10.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 10.0]], dtype=torch.float64
)
th_pbc = torch.tensor([True, False, True])

system_th = System("nm", th_types, th_positions, th_cell, th_pbc)
print("backend", system_th.arrays_backend)
print("positions type", type(system_th.positions), system_th.positions.dtype)
print(system_th.positions)
assert system_th.arrays_backend == "torch"
assert isinstance(system_th.positions, torch.Tensor)
assert float(system_th.positions[3, 0]) == 10.0

backend torch
positions type <class 'torch.Tensor'> torch.float64
tensor([[ 1.,  2.,  3.],
        [ 4.,  5.,  6.],
        [ 7.,  8.,  9.],
        [10., 11., 12.]], dtype=torch.float64)


In [4]:
jx_types = jnp.array([1, 4, 7, 10], dtype=jnp.int32)
jx_positions = jnp.array(
    [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0], [10.0, 11.0, 12.0]],
    dtype=jnp.float64,
)
jx_cell = jnp.array(
    [[10.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 10.0]], dtype=jnp.float64
)
jx_pbc = jnp.array([True, False, True])

system_jx = System("nm", jx_types, jx_positions, jx_cell, jx_pbc)
print("backend", system_jx.arrays_backend)
print("positions type", type(system_jx.positions), system_jx.positions.dtype)
print(system_jx.positions)
assert system_jx.arrays_backend == "jax"
assert isinstance(system_jx.positions, jax.Array)
assert float(system_jx.positions[3, 0]) == 10.0

backend jax
positions type <class 'jaxlib._jax.ArrayImpl'> float64
[[ 1.  2.  3.]
 [ 4.  5.  6.]
 [ 7.  8.  9.]
 [10. 11. 12.]]


## 2. `torch.compile` on extracted arrays

Read `positions` on the host, then compile a kernel that only sees tensors.

In [5]:
def shift_positions(positions):
    return positions + 0.1


extracted = system_th.positions.clone() if system_th.positions.is_leaf else system_th.positions
# getters return read-only DLPack views; copy before mutating / compiling
extracted = system_th.positions.detach().clone()

compiled_shift = torch.compile(shift_positions)
out = compiled_shift(extracted)
print("compiled output[0]", out[0])
print("matches eager", torch.allclose(out, shift_positions(extracted)))

compiled_fullgraph = torch.compile(shift_positions, fullgraph=True)
out_fg = compiled_fullgraph(extracted)
print("fullgraph output[0]", out_fg[0])

compiled output[0] tensor([1.1000, 2.1000, 3.1000], dtype=torch.float64)
matches eager True
fullgraph output[0] tensor([1.1000, 2.1000, 3.1000], dtype=torch.float64)


## 3. `torch.compile` with `System` inside the compiled region

`fullgraph=True` after `dynamo.reset()` is the strict test. Default `torch.compile` may still return correct numbers by graph-breaking back to eager ctypes.

In [6]:
def energy_from_system_positions(system):
    positions = system.positions
    return (positions * positions).sum()


def construct_system_inside(types, positions, cell, pbc):
    system = System("nm", types, positions, cell, pbc)
    return system.positions.sum()


dynamo.reset()

report(
    "eager System.positions sum",
    lambda: float(energy_from_system_positions(system_th)),
)

report(
    "torch.compile(System.positions, default)",
    lambda: float(torch.compile(energy_from_system_positions)(system_th)),
)

report(
    "torch.compile(System.positions, fullgraph=True) after reset",
    lambda: float(
        torch.compile(energy_from_system_positions, fullgraph=True)(system_th)
    ),
)

report(
    "torch.compile(construct System, fullgraph=True)",
    lambda: float(
        torch.compile(construct_system_inside, fullgraph=True)(
            th_types.detach().clone(),
            th_positions.detach().clone(),
            th_cell.detach().clone(),
            th_pbc.detach().clone(),
        )
    ),
)

system_x2 = System(
    "nm",
    th_types.detach().clone(),
    (th_positions * 2).contiguous(),
    th_cell.detach().clone(),
    th_pbc.detach().clone(),
)
compiled_default = torch.compile(energy_from_system_positions)
print(
    "default compile two systems:",
    float(compiled_default(system_th)),
    float(compiled_default(system_x2)),
    "(eager would be 650 and 2600)",
)

OK    eager System.positions sum: 650.0


OK    torch.compile(System.positions, default): 650.0
OK    torch.compile(System.positions, fullgraph=True) after reset: 650.0
FAIL  torch.compile(construct System, fullgraph=True)
      Unsupported: Unsupported method call
  Explanation: Dynamo does not know how to trace method `__new__` of class `PyCPointerType`
  Hint: Avoid calling `PyCPointerType.__new__` in your code.
  Hint: Please report an issue to PyTorch.

  Developer debug context: call_method UserDefinedClassVariable(<class '_ctypes._Pointer'>) __new__ [UserDefinedClassVariable(<class 'ctypes.LP_mta_system_t'>)] {}

 For more details about this graph break, please visit: https://meta-pytorch.github.io/compile-graph-break-site/gb/gb0156.html

from user code:
   File "/var/folders/wy/kcg4xf054_v1h2zwbhjb2g_w0000gn/T/ipykernel_64722/3915876863.py", line 7, in construct_system_inside
    system = System("nm", types, positions, cell, pbc)
  File "/Users/ericboittier/metawork/metatomic/.tox/core-tests/lib/python3.14/site-package

## 4. `jax.jit` on extracted arrays

In [7]:
def jax_shift(positions):
    return positions + 0.1


extracted_jx = jnp.array(system_jx.positions)
jitted_shift = jax.jit(jax_shift)
out_jx = jitted_shift(extracted_jx)
print("jitted output[0]", out_jx[0])
print("matches eager", jnp.allclose(out_jx, jax_shift(extracted_jx)))

jitted output[0] [1.1 2.1 3.1]
matches eager True


## 5. `jax.jit` with `System` inside the traced region

`System` is not a JAX pytree; ctypes getters are Python side effects.

In [8]:
report(
    "eager jax System.positions sum",
    lambda: float(system_jx.positions.sum()),
)

report(
    "jax.jit(fn(system))",
    lambda: float(jax.jit(lambda system: system.positions.sum())(system_jx)),
)

report(
    "jax.jit(construct System)",
    lambda: float(
        jax.jit(construct_system_inside)(jx_types, jx_positions, jx_cell, jx_pbc)
    ),
)

report(
    "jax.jit(read system.positions from closure)",
    lambda: float(jax.jit(lambda: system_jx.positions.sum())()),
)

OK    eager jax System.positions sum: 78.0
FAIL  jax.jit(fn(system))
      TypeError: Error interpreting argument to <function <lambda>.<locals>.<lambda> at 0x11f1f2e50> as an abstract array. The problematic value is of type <class 'metatomic.System'> and was passed to the function at path system.
This typically means that a jit-wrapped function was called with a non-array argument, and this argument was not marked as static using the static_argnums or static_argnames parameters of jax.jit.
FAIL  jax.jit(construct System)
      RuntimeError: __dlpack__ call failed
OK    jax.jit(read system.positions from closure): 78.0


78.0

## Conclusion

Results from this run (torch 2.13.0, jax 0.11.1):

- **Host construction / getters** work for numpy, torch, and jax (DLPack round-trip).
- **Extracted arrays** work with `torch.compile` (including `fullgraph=True`) and `jax.jit`.
- **`System` is not a compiled-region type.** Constructing `System` inside `torch.compile(..., fullgraph=True)` fails: Dynamo cannot trace `ctypes.POINTER(...)()` (`PyCPointerType.__new__`). Reading `system.positions` under default `torch.compile` still returns the right numbers because ctypes graph-breaks back to eager Python (650 vs 2600 for a 2× scaled system). Do not rely on Dynamo tracing the getter.
- **`jax.jit`** rejects `System` as a non-array argument. Constructing `System` inside `jit` fails (`__dlpack__ call failed`). A closure that reads `system.positions` at trace time can look like it works, but that freezes host Python state into the jaxpr — it is not tracing the wrapper.

`metatomic.System` is a host-side ctypes wrapper. Build it in Python, then pass arrays into compiled kernels. TorchScript models should keep using `metatomic.torch.System`.